In [11]:
import json
import pandas as pd
from datetime import datetime
from pathlib import Path

with open(r"D:\progscode_2semestr\python_programming\data\raw\JP_TYO\JP_TYO_20260323_215719.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data.keys())
print(str(data)[:500]) 


hourly = data["hourly"]

df = pd.DataFrame({
    "ts": hourly["time"],
    "temperature_2m": hourly["temperature_2m"],
    "relative_humidity_2m": hourly["relative_humidity_2m"],
    "precipitation": hourly["precipitation"],
    "wind_speed_10m": hourly["wind_speed_10m"],
})

df["city_id"] = "JP_TYO"

print(df.head())
print(df.shape)
print(df.dtypes)

df["ts"] = pd.to_datetime(df["ts"])

cols = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

df = df.drop_duplicates(subset=["city_id", "ts"])

df = df[
    (df["temperature_2m"].between(-80, 60)) &
    (df["relative_humidity_2m"].between(0, 100)) &
    (df["precipitation"] >= 0)
]

df = df.dropna(subset=["ts"])

print(df.head())
print(df.dtypes)
print(df.isna().sum())
print("rows:", len(df))

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
normalized_dir = Path(r"D:\progscode_2semestr\python_programming\data\normalized\variant_06")
normalized_dir.mkdir(parents=True, exist_ok=True)
csv_file = normalized_dir / f"{timestamp}.csv"
df.to_csv(csv_file, index=False)

<class 'dict'>
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
{'latitude': 35.676624, 'longitude': 139.69112, 'generationtime_ms': 9.584426879882812, 'utc_offset_seconds': 32400, 'timezone': 'Asia/Tokyo', 'timezone_abbreviation': 'GMT+9', 'elevation': 37.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'wind_speed_10m': 'km/h'}, 'hourly': {'time': ['2026-02-01T00:00', '2026-02-01T01:00', '2026-02-01T02:00', '2026-02-01T03:00', '2026-02-01T04:00', '2026-02-01T05:00', '2026-02-01T06:00', '2026
                 ts  temperature_2m  relative_humidity_2m  precipitation  \
0  2026-02-01T00:00             2.0                    43            0.0   
1  2026-02-01T01:00             1.6                    44            0.0   
2  2026-02-01T02:00             1.4                    45            0.0   
3  2026-02-01T03:00     